So far we've learnt how to scrape the web, and how to make a request for information from an API. Some websites make APIs even easier. Check out [RapidAPI](https://rapidapi.com/) they take care of writing most of the code for you.

We will use the [AeroDataBox API](https://rapidapi.com/aedbx-aedbx/api/aerodatabox/), which can retrieve all sorts of information about flights and airports. We will show you how to retrieve information about the airports, and then it's up to you to apply this, along with what you've already learnt this week, to **produce a function, which retrieves tomorrows flight information for the major airports in the cities you web scraped**.

In [42]:
import requests
import pandas as pd

In [43]:
rapidapi_key = "400f91c72amshb5a4a830ea0b7bfp1617c4jsn75edb7b0ad43"

In [44]:
with open(".env", "w") as f:
    f.write("CON_STRING=mysql+pymysql://root:YOUR_PASSWORD/gans\n")
    f.write("OPENWEATHER_KEY=your_api_key_here\n")
    f.write("RAPIDAPI_KEY=xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx\n")

We'll simulate a DataFrame like the kind hosted on our sample database. You should use the table on your own database to find airports for those cities.

In [45]:
cities_df = pd.DataFrame({
    "city_id": [4, 5, 6],
    "name": ["Berlin", "Paris", "London"],
    "country": ["Germany", "France", "England"],
    "latitude": [52.5200, 48.8567, 51.5072],
    "longitude": [13.4050, 2.3522, -0.1275]
})

cities_df

,city_id,name,country,latitude,longitude
0,4,Berlin,Germany,52.5200,13.4050
1,5,Paris,France,48.8567,2.3522
2,6,London,England,51.5072,-0.1275


On the left hand side of the AeroDataBox API page, you'll see a list of options for information that you can retrieve:
> - Flight API
> - Flight Alert API
> - Airport API
> - Aircraft API
> - Industry API
> - Statistical API
> - Miscellaneous API
> - Healthcheck & Status API

1. We want to select `Airport API`

2. Then within Airport API we want to select `Search airports by location`

3. Now in the middle third you'll want to select `Params` and enter the `latitude` and `longitude` of any city to test... we chose Berlin: latitude 52.52 longitude 13.405. The default `radiusKM` of 50km seems fine. And finally set `withFlightInfoOnly` to true, so it will only return airports which have flight data (scheduled or live) available.

4. On the right hand third of the screen you should see a block of code that looks pretty unfamiliar. This is because by default the code is probably set to *(Shell) Curl*. However, we have the power to change this to familiar python. Select the "Target" dropdown box at the top of the code and select `python ` and the "Client" dropdown to `Requests`.

Now you can copy the code to your notebook and it should look a little something like the cell below:

In [46]:
import requests

url = "https://aerodatabox.p.rapidapi.com/airports/search/location"

querystring = {"lat":"52.52","lon":"13.405","radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}

headers = {
    "x-rapidapi-key": rapidapi_key, # it seems they've actually left this out right now
	"x-rapidapi-host": "aerodatabox.p.rapidapi.com",
	"Content-Type": "application/json"
}

response = requests.get(url, headers=headers, params=querystring)

print(response.json())

{'searchBy': {'lat': 52.52, 'lon': 13.405}, 'count': 2, 'items': [{'icao': 'EDDT', 'iata': 'TXL', 'name': 'Berlin -Tegel', 'shortName': '-Tegel', 'municipalityName': 'Berlin', 'location': {'lat': 52.5597, 'lon': 13.287699}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}, {'icao': 'EDDB', 'iata': 'BER', 'name': 'Berlin Brandenburg', 'shortName': 'Brandenburg', 'municipalityName': 'Berlin', 'location': {'lat': 52.35139, 'lon': 13.493889}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}]}


We can now turn this into a dataframe using `.json_normalize()`

In [47]:
pd.json_normalize(response.json()['items'])

,icao,iata,name,shortName,municipalityName,countryCode,timeZone,location.lat,location.lon
0,EDDT,TXL,Berlin -Tegel,-Tegel,Berlin,DE,Europe/Berlin,52.55970,13.287699
1,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889


In [48]:
print(response.json())

{'searchBy': {'lat': 52.52, 'lon': 13.405}, 'count': 2, 'items': [{'icao': 'EDDT', 'iata': 'TXL', 'name': 'Berlin -Tegel', 'shortName': '-Tegel', 'municipalityName': 'Berlin', 'location': {'lat': 52.5597, 'lon': 13.287699}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}, {'icao': 'EDDB', 'iata': 'BER', 'name': 'Berlin Brandenburg', 'shortName': 'Brandenburg', 'municipalityName': 'Berlin', 'location': {'lat': 52.35139, 'lon': 13.493889}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}]}


It looks like this isn't perfect: Tegel is retired since 2021. Let's keep in mind that we'll need to account for this somehow.

Let's now use this to find the airports around multiple cities

In [49]:
def get_airports(cities_df):
    # API headers
    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
    url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
    

    # DataFrame to store results
    all_airports = []

    for _, row in cities_df.iterrows():
    # Construct the URL with the latitude and longitude
        querystring = {"lat":row["latitude"],"lon":row["longitude"],"radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}

        # Make the API request
        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code == 200:
            data = response.json()
            airports = pd.json_normalize(data.get('items', []))
            all_airports.append(airports)
        else:
            print(f"WARNING: Failed to retrieve airports for {row["name"]}")

    return pd.concat(all_airports, ignore_index=True)

In [50]:
get_airports(cities_df)

,icao,iata,name,shortName,municipalityName,countryCode,timeZone,location.lat,location.lon
0,EDDT,TXL,Berlin -Tegel,-Tegel,Berlin,DE,Europe/Berlin,52.55970,13.287699
1,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889


## **Challenge:** Arrivals information
Using what you have been shown above, plus the skills you've learnt in the last couple of days:
1. In `AeroDataBox API` use the `Flight API` > `FIDS/Schedules: Airport departures and arrivals (by time range)` section
2. Fill out the parameters in the middle third and then copy the `python: requests` code from the right hand third
3. Explore the data you get back. What would be useful in your DataFrame and what can be excluded? Remember Gans wants to know about when people are arriving in the city
4. Make a DataFrame from the information you see as important
5. Condense everything you did above into a function that can take a list of ICAO codes as an input, and as an output gives you a DataFrame with the information for *tomorrows arrivals*

### 
_____

We can start by loading our needed secrets and reading the `'cities'` table from Gans's database.

In [51]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

In [52]:
with open(".env", "w") as f:
    f.write("CON_STRING=mysql+pymysql://root:YOUR_PASSWORD/gans\n")
    f.write("OPENWEATHER_KEY=your_api_key_here\n")
    f.write("RAPIDAPI_KEY=xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx\n")

In [53]:
load_dotenv()
connection_string = os.getenv("CON_STRING")
rapidapi_key = os.getenv("RAPIDAPI_KEY")
cities_df = pd.read_sql("cities", con=connection_string)
cities_df

,city_id,name,country,latitude,longitude
0,1,Berlin,Germany,52.5200,13.405
1,2,Hamburg,Germany,53.5500,10.000
2,3,Munich,Germany,48.1375,11.575


### Step 1: Airports

A function for finding airports is already provided. We'll modify it just a little, selecting only a few columns from each result and renaming one of them.

In [54]:
def get_airports(cities_df):
    # API constants
    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
    url = "https://aerodatabox.p.rapidapi.com/airports/search/location"

    # DataFrame to store results
    all_airports = []

    for _, row in cities_df.iterrows():
        querystring = {"lat":row["latitude"],"lon":row["longitude"],"radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}
        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code == 200:
            data = response.json()
            airports = pd.json_normalize(data.get('items', []))
            airports["city_id"] = row["city_id"] # add city_id for foreign key reference
            all_airports.append(airports)
        else:
            print(f"WARNING: Failed to retrieve airports for {row["name"]}")

    airports_df = pd.concat(all_airports, ignore_index=True) # make one DataFrame from individual results
    airports_df = airports_df[["icao", "name", "city_id"]] 

    return airports_df

In [55]:
airports_df = get_airports(cities_df)
airports_df

,icao,name,city_id
0,EDDT,Berlin -Tegel,1
1,EDDB,Berlin Brandenburg,1


This result looks just fine. Before proceeding with flights, let's write a table definition to `03_gans_schema.sql` and populate the table with `airports_df`. One thing worth noting, we'll add an extra column for `'active'` to the SQL table. This way if AeroDataBox provides a bad airport (like Tegel) we have a way to exclude it from flight searches. Equally valid, we could work out a method to blacklist certain airports from `airports_df`.

In [57]:
airports_df.to_sql(
    "airports",
    con=connection_string,
    if_exists="append",
    index=False
)

IntegrityError: (pymysql.err.IntegrityError) (1062, "Duplicate entry 'EDDT' for key 'airports.PRIMARY'")
[SQL: INSERT INTO airports (icao, name, city_id) VALUES (%(icao)s, %(name)s, %(city_id)s)]
[parameters: [{'icao': 'EDDT', 'name': 'Berlin -Tegel', 'city_id': 1}, {'icao': 'EDDB', 'name': 'Berlin Brandenburg', 'city_id': 1}]]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

### Step 2: Flights

Let's begin by reading from the database's `'airports'` table. Here, we select only airports that we've flagged as "active".

In [74]:
airports_df = pd.read_sql("SELECT * FROM airports WHERE `active` = 1", con=connection_string)
airports_df

,icao,name,active,city_id
0,EDDB,Berlin Brandenburg,1,1


#### One airport, one time window

Following the approach taken with weather forecasts, we'll start small with a single call to the API and then add layers of loops to retrieve more and more data.

In [75]:
# copy headers from above
headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }

# copy url and querystring from AeroDataBox website, filling in one icao code
url = "https://aerodatabox.p.rapidapi.com/flights/airports/icao/EDDB/2026-08-22T00:00/2026-08-22T11:59"
querystring = {"withLeg":"false","direction":"Arrival","withCancelled":"false","withCodeshared":"false"}

response = requests.get(url, headers=headers, params=querystring)
print(response.json())

{'arrivals': [{'movement': {'airport': {'icao': 'LTFM', 'iata': 'IST', 'name': 'Istanbul', 'countryCode': 'tr', 'timeZone': 'Europe/Istanbul'}, 'scheduledTime': {'utc': '2026-08-22 03:40Z', 'local': '2026-08-22 05:40+02:00'}, 'revisedTime': {'utc': '2026-08-22 03:26Z', 'local': '2026-08-22 05:26+02:00'}, 'runwayTime': {'utc': '2026-08-22 03:26Z', 'local': '2026-08-22 05:26+02:00'}, 'terminal': '1', 'gate': 'X10', 'baggageBelt': 'A1', 'runway': '25L', 'quality': ['Basic', 'Live']}, 'number': 'TK 1220', 'callSign': 'THY9XT', 'status': 'Arrived', 'codeshareStatus': 'IsOperator', 'isCargo': False, 'aircraft': {'reg': 'TC-LPC', 'modeS': '4BB203', 'model': 'Airbus A321 NEO'}, 'airline': {'name': 'Turkish', 'iata': 'TK', 'icao': 'THY'}}, {'movement': {'airport': {'icao': 'LTAI', 'iata': 'AYT', 'name': 'Antalya', 'countryCode': 'tr', 'timeZone': 'Europe/Istanbul'}, 'scheduledTime': {'utc': '2026-08-22 04:05Z', 'local': '2026-08-22 06:05+02:00'}, 'revisedTime': {'utc': '2026-08-22 03:27Z', 'loc

We can now begin to explore the response

In [76]:
response_json = response.json()
response_json.keys()

dict_keys(['arrivals'])

In [77]:
# looks like a list
response_json["arrivals"]

[{'movement': {'airport': {'icao': 'LTFM',
    'iata': 'IST',
    'name': 'Istanbul',
    'countryCode': 'tr',
    'timeZone': 'Europe/Istanbul'},
   'scheduledTime': {'utc': '2026-08-22 03:40Z',
    'local': '2026-08-22 05:40+02:00'},
   'revisedTime': {'utc': '2026-08-22 03:26Z',
    'local': '2026-08-22 05:26+02:00'},
   'runwayTime': {'utc': '2026-08-22 03:26Z',
    'local': '2026-08-22 05:26+02:00'},
   'terminal': '1',
   'gate': 'X10',
   'baggageBelt': 'A1',
   'runway': '25L',
   'quality': ['Basic', 'Live']},
  'number': 'TK 1220',
  'callSign': 'THY9XT',
  'status': 'Arrived',
  'codeshareStatus': 'IsOperator',
  'isCargo': False,
  'aircraft': {'reg': 'TC-LPC', 'modeS': '4BB203', 'model': 'Airbus A321 NEO'},
  'airline': {'name': 'Turkish', 'iata': 'TK', 'icao': 'THY'}},
 {'movement': {'airport': {'icao': 'LTAI',
    'iata': 'AYT',
    'name': 'Antalya',
    'countryCode': 'tr',
    'timeZone': 'Europe/Istanbul'},
   'scheduledTime': {'utc': '2026-08-22 04:05Z',
    'local'

In [78]:
# try to understand one arrival
# check the webpage as well, there may be items not present in every arrival!
arrival = response_json["arrivals"][0]
arrival

{'movement': {'airport': {'icao': 'LTFM',
   'iata': 'IST',
   'name': 'Istanbul',
   'countryCode': 'tr',
   'timeZone': 'Europe/Istanbul'},
  'scheduledTime': {'utc': '2026-08-22 03:40Z',
   'local': '2026-08-22 05:40+02:00'},
  'revisedTime': {'utc': '2026-08-22 03:26Z',
   'local': '2026-08-22 05:26+02:00'},
  'runwayTime': {'utc': '2026-08-22 03:26Z',
   'local': '2026-08-22 05:26+02:00'},
  'terminal': '1',
  'gate': 'X10',
  'baggageBelt': 'A1',
  'runway': '25L',
  'quality': ['Basic', 'Live']},
 'number': 'TK 1220',
 'callSign': 'THY9XT',
 'status': 'Arrived',
 'codeshareStatus': 'IsOperator',
 'isCargo': False,
 'aircraft': {'reg': 'TC-LPC', 'modeS': '4BB203', 'model': 'Airbus A321 NEO'},
 'airline': {'name': 'Turkish', 'iata': 'TK', 'icao': 'THY'}}

In [79]:
arrival.keys()

dict_keys(['movement', 'number', 'callSign', 'status', 'codeshareStatus', 'isCargo', 'aircraft', 'airline'])

In [80]:
arrival_dict = {
    "depart_airport": arrival["movement"]["airport"]["name"],
    "depart_country": arrival["movement"]["airport"]["countryCode"].upper(), # country codes are more often all capital letters
    "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
    "arrive_time_revised": arrival["movement"]["revisedTime"]["local"],
    "flight_number": arrival["number"],
    "aircraft": arrival["aircraft"]["model"]
}
arrival_dict

{'depart_airport': 'Istanbul',
 'depart_country': 'TR',
 'arrive_time_scheduled': '2026-08-22 05:40+02:00',
 'arrive_time_revised': '2026-08-22 05:26+02:00',
 'flight_number': 'TK 1220',
 'aircraft': 'Airbus A321 NEO'}

Now let's see if this works with the rest of the flights in the response.

In [82]:
arrivals = []
for arrival in response_json["arrivals"]:
    arrival_dict = {
        "depart_airport": arrival["movement"]["airport"].get("name", None),
        "depart_country": arrival["movement"]["airport"].get("countryCode", "XX").upper(),
        "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
        "arrive_time_revised": arrival["movement"].get("revisedTime", {}).get("local", None),
        "flight_number": arrival.get("number", None),
        "aircraft": arrival.get("aircraft", {}).get("model", None)
    }
    arrivals.append(arrival_dict)

flights_df = pd.DataFrame(arrivals)
flights_df

,depart_airport,depart_country,arrive_time_scheduled,arrive_time_revised,flight_number,aircraft
0,Istanbul,TR,2026-08-22 05:40+02:00,2026-08-22 05:26+02:00,TK 1220,Airbus A321 NEO
1,Antalya,TR,2026-08-22 06:05+02:00,2026-08-22 05:27+02:00,XQ 668,Boeing 737-800
2,İzmir,TR,2026-08-22 06:05+02:00,2026-08-22 05:48+02:00,XQ 966,Boeing 737-800
3,Adana,TR,2026-08-22 06:45+02:00,2026-08-22 06:20+02:00,XQ 1774,Boeing 737 MAX 8
4,Beirut,LB,2026-08-22 06:10+02:00,2026-08-22 06:30+02:00,SR 1599,Airbus A320-200
...,...,...,...,...,...,...
75,London,GB,2026-08-22 11:45+02:00,2026-08-22 11:37+02:00,EW 8461,Airbus A320 NEO
76,Tirana,AL,2026-08-22 11:30+02:00,2026-08-22 11:39+02:00,W4 5105,Airbus A321
77,Palma De Mallorca,ES,2026-08-22 12:25+02:00,2026-08-22 11:51+02:00,U2 5120,Airbus A320
78,Beirut,LB,2026-08-22 12:05+02:00,2026-08-22 11:54+02:00,ME 245,Airbus A320


Looks like we get the same problem we saw with weather: not every key is included in every arrival. `.get()` will help us here again.

In [84]:
arrivals = [] # empty list to store arrivals
for arrival in response_json["arrivals"]:
    arrival_dict = {
        "depart_airport": arrival["movement"]["airport"]["name"],
        "depart_country": arrival["movement"]["airport"]["countryCode"].upper(),
        "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
        "arrive_time_revised": arrival["movement"].get("revisedTime", {}).get("local", None), 
        "flight_number": arrival["number"],
        "aircraft": arrival.get("aircraft", {}).get("model", None)
    }
    arrivals.append(arrival_dict)

flights_df = pd.DataFrame(arrivals)
flights_df

,depart_airport,depart_country,arrive_time_scheduled,arrive_time_revised,flight_number,aircraft
0,Istanbul,TR,2026-08-22 05:40+02:00,2026-08-22 05:26+02:00,TK 1220,Airbus A321 NEO
1,Antalya,TR,2026-08-22 06:05+02:00,2026-08-22 05:27+02:00,XQ 668,Boeing 737-800
2,İzmir,TR,2026-08-22 06:05+02:00,2026-08-22 05:48+02:00,XQ 966,Boeing 737-800
3,Adana,TR,2026-08-22 06:45+02:00,2026-08-22 06:20+02:00,XQ 1774,Boeing 737 MAX 8
4,Beirut,LB,2026-08-22 06:10+02:00,2026-08-22 06:30+02:00,SR 1599,Airbus A320-200
...,...,...,...,...,...,...
75,London,GB,2026-08-22 11:45+02:00,2026-08-22 11:37+02:00,EW 8461,Airbus A320 NEO
76,Tirana,AL,2026-08-22 11:30+02:00,2026-08-22 11:39+02:00,W4 5105,Airbus A321
77,Palma De Mallorca,ES,2026-08-22 12:25+02:00,2026-08-22 11:51+02:00,U2 5120,Airbus A320
78,Beirut,LB,2026-08-22 12:05+02:00,2026-08-22 11:54+02:00,ME 245,Airbus A320


Our API will only let us retrieve flight data for up to 12 hours at a time. To cover all of tomorrow, we'll need to build a `for` loop. We'll also need to work out how to get tomorrow's date.

#### One airport, two time windows

In [85]:
today = pd.Timestamp.now().date()
one_day = pd.Timedelta(1, "day")
tomorrow = today+one_day
tomorrow_str = tomorrow.strftime("%Y-%m-%d")
tomorrow_str

'2026-08-25'

In [86]:
headers = { # headers stay constant
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
times = [["00:00", "11:59"], ["12:00", "23:59"]]
airport = airports_df.loc[0] # use the DataFrame this time, instead of copying call details from website

arrivals = [] # empty list to store arrivals
for start_time, end_time in times: # we can "unpack" the interior lists into two iteration variables
    url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{airport["icao"]}/{tomorrow_str}T{start_time}/{tomorrow_str}T{end_time}"
    querystring = {"withLeg":"false","direction":"Arrival","withCancelled":"false","withCodeshared":"false"}
    response = requests.get(url, headers=headers, params=querystring)

    if response.status_code == 200:
        data = response.json()["arrivals"]
        for arrival in data:
            arrival_dict = {
                "arrive_airport": airport["icao"], # add foreign-key information
                "depart_airport": arrival["movement"]["airport"]["name"],
                "depart_country": arrival["movement"]["airport"]["countryCode"].upper(),
                "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
                "arrive_time_revised": arrival["movement"].get("revisedTime", {}).get("local", None), 
                "flight_number": arrival["number"],
                "aircraft": arrival["aircraft"]["model"]
            }
            arrivals.append(arrival_dict)
        
flights_df = pd.DataFrame(arrivals)
flights_df

,arrive_airport,depart_airport,depart_country,arrive_time_scheduled,arrive_time_revised,flight_number,aircraft
0,EDDB,İzmir,TR,2026-08-25 06:05+02:00,2026-08-25 06:05+02:00,XQ 966,Boeing 737-700 (winglets)
1,EDDB,Beirut,LB,2026-08-25 06:10+02:00,2026-08-25 06:10+02:00,SR 1501,Airbus A320
2,EDDB,Gaziantep,TR,2026-08-25 06:45+02:00,2026-08-25 06:45+02:00,XQ 1766,Boeing 737 MAX 8
3,EDDB,Bucharest,RO,2026-08-25 07:00+02:00,2026-08-25 07:00+02:00,W4 3109,Airbus A321
4,EDDB,Newark,US,2026-08-25 07:15+02:00,2026-08-25 07:15+02:00,UA 962,Boeing 767-300 (winglets)
...,...,...,...,...,...,...,...
76,EDDB,London,GB,2026-08-25 11:45+02:00,2026-08-25 11:45+02:00,EW 8461,Airbus A320 NEO
77,EDDB,Istanbul,TR,2026-08-25 11:50+02:00,2026-08-25 11:50+02:00,TK 1729,Airbus A330-300
78,EDDB,Paris,FR,2026-08-25 11:55+02:00,2026-08-25 11:55+02:00,AF 1734,Airbus A220-300
79,EDDB,Nice,FR,2026-08-25 11:55+02:00,2026-08-25 11:55+02:00,EW 8427,Airbus A320-200 (sharklets)


Oh no, there's even more keys missing! We'll add some more `.get()` statements.

In [87]:
headers = { # headers stay constant
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
times = [["00:00", "11:59"], ["12:00", "23:59"]]
airport = airports_df.loc[0] # use the DataFrame this time, instead of copying call details from website

arrivals = [] # empty list to store arrivals
for start_time, end_time in times: # we can "unpack" the interior lists into two iteration variables
    url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{airport["icao"]}/{tomorrow_str}T{start_time}/{tomorrow_str}T{end_time}"
    querystring = {"withLeg":"false","direction":"Arrival","withCancelled":"false","withCodeshared":"false"}
    response = requests.get(url, headers=headers, params=querystring)

    if response.status_code == 200:
        data = response.json()["arrivals"]
        for arrival in data:
            arrival_dict = {
                "arrive_icao": airport["icao"], # add foreign-key information
                "depart_icao": arrival["movement"]["airport"].get("icao", None), # add this since countryCode and maybe name aren't reliable
                "depart_airport": arrival["movement"]["airport"].get("name", None),
                "depart_country": arrival["movement"]["airport"].get("countryCode", "XX").upper(),
                "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
                "arrive_time_revised": arrival["movement"].get("revisedTime", {}).get("local", None), 
                "flight_number": arrival.get("number", None),
                "aircraft": arrival.get("aircraft", {}).get("model", None)
            }
            arrivals.append(arrival_dict)
        
flights_df = pd.DataFrame(arrivals)
flights_df

,arrive_icao,depart_icao,depart_airport,depart_country,arrive_time_scheduled,arrive_time_revised,flight_number,aircraft
0,EDDB,LFMN,Nice,FR,2026-08-25 12:00+02:00,2026-08-25 12:00+02:00,U2 5144,Airbus A320-200 (sharklets)
1,EDDB,EDDM,Munich,DE,2026-08-25 12:05+02:00,2026-08-25 12:05+02:00,LH 1926,Airbus A321-100
2,EDDB,OLBA,Beirut,LB,2026-08-25 12:05+02:00,2026-08-25 12:05+02:00,ME 245,Airbus A320-200 (sharklets)
3,EDDB,LEPA,Palma De Mallorca,ES,2026-08-25 12:05+02:00,2026-08-25 12:05+02:00,FR 227,Boeing 737-800 (winglets)
4,EDDB,ENBR,Bergen,NO,2026-08-25 12:10+02:00,2026-08-25 12:10+02:00,DY 1122,Boeing 737-800 (winglets)
...,...,...,...,...,...,...,...,...
156,EDDB,LPPR,Porto,PT,2026-08-25 23:00+02:00,2026-08-25 23:00+02:00,FR 2945,Boeing 737 MAX 8
157,EDDB,LIME,Bergamo,IT,2026-08-25 23:00+02:00,2026-08-25 23:00+02:00,FR 2679,Boeing 737-800 (winglets)
158,EDDB,EGSS,London,GB,2026-08-25 23:00+02:00,2026-08-25 23:00+02:00,FR 176,Boeing 737 MAX 8
159,EDDB,LEMD,Madrid,ES,2026-08-25 23:10+02:00,2026-08-25 23:10+02:00,IB 787,Airbus A321-100


#### All airports, two time windows

In [89]:
headers = { # headers stay constant
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
times = [["00:00", "11:59"], ["12:00", "23:59"]]

arrivals = [] # empty list to store arrivals
for _, airport in airports_df.iterrows():
    for start_time, end_time in times: # we can "unpack" the interior lists into two iteration variables
        url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{airport["icao"]}/{tomorrow_str}T{start_time}/{tomorrow_str}T{end_time}"
        querystring = {"withLeg":"false","direction":"Arrival","withCancelled":"false","withCodeshared":"false"}
        response = requests.get(url, headers=headers, params=querystring)
    
        if response.status_code == 200:
            data = response.json()["arrivals"]
            for arrival in data:
                arrival_dict = {
                    "arrive_icao": airport["icao"], # add foreign-key information
                    "depart_icao": arrival["movement"]["airport"].get("icao", None), # add this since countryCode and maybe name aren't reliable
                    "depart_airport": arrival["movement"]["airport"].get("name", None),
                    "depart_country": arrival["movement"]["airport"].get("countryCode", None), # move .upper() to deal with NoneTypes
                    "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
                    "arrive_time_revised": arrival["movement"].get("revisedTime", {}).get("local", None), 
                    "flight_number": arrival.get("number", None),
                    "aircraft": arrival.get("aircraft", {}).get("model", None)
                }
                arrivals.append(arrival_dict)
        
flights_df = pd.DataFrame(arrivals)
flights_df["depart_country"] = flights_df["depart_country"].str.upper()
flights_df

,arrive_icao,depart_icao,depart_airport,depart_country,arrive_time_scheduled,arrive_time_revised,flight_number,aircraft
0,EDDB,LTBJ,İzmir,TR,2026-08-25 06:05+02:00,2026-08-25 06:05+02:00,XQ 966,Boeing 737-700 (winglets)
1,EDDB,OLBA,Beirut,LB,2026-08-25 06:10+02:00,2026-08-25 06:10+02:00,SR 1501,Airbus A320
2,EDDB,LTAJ,Gaziantep,TR,2026-08-25 06:45+02:00,2026-08-25 06:45+02:00,XQ 1766,Boeing 737 MAX 8
3,EDDB,LROP,Bucharest,RO,2026-08-25 07:00+02:00,2026-08-25 07:00+02:00,W4 3109,Airbus A321
4,EDDB,KEWR,Newark,US,2026-08-25 07:15+02:00,2026-08-25 07:15+02:00,UA 962,Boeing 767-300 (winglets)
...,...,...,...,...,...,...,...,...
76,EDDB,EGLL,London,GB,2026-08-25 11:45+02:00,2026-08-25 11:45+02:00,EW 8461,Airbus A320 NEO
77,EDDB,LTFM,Istanbul,TR,2026-08-25 11:50+02:00,2026-08-25 11:50+02:00,TK 1729,Airbus A330-300
78,EDDB,LFMN,Nice,FR,2026-08-25 11:55+02:00,2026-08-25 11:55+02:00,EW 8427,Airbus A320-200 (sharklets)
79,EDDB,LFPG,Paris,FR,2026-08-25 11:55+02:00,2026-08-25 11:55+02:00,AF 1734,Airbus A220-300


Success! Now add a table definition to the database before pushing the data.

In [90]:
# we get "Incorrect datetime value: '2026-08-22 05:40+02:00' for column 'arrive_time_scheduled' at row 1", so let's convert to datetime
flights_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   arrive_icao            81 non-null     object
 1   depart_icao            81 non-null     object
 2   depart_airport         81 non-null     object
 3   depart_country         81 non-null     object
 4   arrive_time_scheduled  81 non-null     object
 5   arrive_time_revised    78 non-null     object
 6   flight_number          81 non-null     object
 7   aircraft               81 non-null     object
dtypes: object(8)
memory usage: 5.2+ KB


In [91]:
flights_df["arrive_time_scheduled"] = pd.to_datetime(flights_df["arrive_time_scheduled"])
flights_df["arrive_time_revised"] = pd.to_datetime(flights_df["arrive_time_revised"])

In [92]:
flights_df.to_sql(
    "flights",
    con=connection_string,
    if_exists="append",
    index=False
)

81

In [93]:
flights_df["arrive_time_scheduled"] = pd.to_datetime(flights_df["arrive_time_scheduled"])
flights_df["arrive_time_revised"] = pd.to_datetime(flights_df["arrive_time_revised"])

In [94]:
flights_df.to_sql(
    "flights",
    con=connection_string,
    if_exists="append",
    index=False
)

81